# Donation-Intent Classifier Training — v4: Attention Pooling + Class-Balanced Focal Loss + Supervised Contrastive Regularisation

This notebook is a from-scratch fork of `capstone-v3-augmented.ipynb`
(kept untouched as the baseline). Same data, same 3-encoder benchmark
(RoBERTa-base / DeBERTa-v3-base / TOD-BERT), same speaker-role encoding
and EDA augmentation. **What changed is the model architecture, the loss,
and one training-loop bug** — all aimed at one observed failure mode:

> The v3 model tends to collapse toward predicting the majority class
> (`binary_label="yes"`, `modifier="none"`) instead of learning the
> minority classes, even with inverse-frequency class weights in the loss.

## Root cause found while reading v3's training loop

`train_one_encoder()` picks its "best" checkpoint (and its early-stopping
signal) using:

```python
val_score = (val_metrics["binary_f1"] + val_metrics["modifier_macro_f1_yes"]) / 2
```

but `val_metrics["binary_f1"]` is computed with
`f1_score(..., average="binary", pos_label=yes_idx)` — **the F1 of the
`yes` class only**, not macro-F1 across both classes. A model that always
predicts `yes` gets 100% recall and a *high* score on this exact metric,
because it never counts `no`-class recall. So the metric used to decide
which checkpoint to keep can reward the majority-collapse behaviour it
should be penalising. (The separate `find_optimal_binary_threshold()`
function already searches for macro-F1 — only the epoch-to-epoch
checkpoint selection had the bug.)

## v4 changes

1. **Fix: checkpoint selection now uses binary macro-F1**, not yes-only F1.
   Free — a one-line change — but likely the single highest-leverage fix
   here, since it changes *what "best" means* during training.
2. **Class-Balanced Focal Loss** (Cui et al., 2019, CVPR — "effective
   number of samples") replaces plain inverse-frequency-weighted
   cross-entropy + label smoothing on both heads. Focal loss's
   `(1-p_t)^gamma` term keeps down-weighting *easy, already-confident*
   majority-class predictions throughout training, instead of a single
   static class weight that a model can learn to shrug off.
3. **Attention pooling** replaces masked mean-pooling. A learned
   `tanh`-attention score per token lets the model weight the few tokens
   that actually carry modifier signal (`"if"`, `"next month"`, `"maybe"`,
   negations) instead of diluting them across a whole dialogue's worth of
   mean-pooled tokens. Fused with the `[CLS]`/`<s>` vector.
4. **Cascaded modifier head**: the modifier head now takes
   `[shared_repr ; softmax(binary_logits).detach()]` as input, mirroring
   the task's actual structure (modifier is only meaningful when
   `binary_label == "yes"`) instead of being a plain sibling head that
   only shares the pooled vector.
5. **Supervised Contrastive auxiliary loss** (Khosla et al., 2020) computed
   in-batch on the pooled representation — a lightweight, single-stage
   stand-in for README idea #3 (full SimCSE needs a separate unsupervised
   pretraining stage, which does not fit a "~20-30 min" budget). Adds
   near-zero compute (one `B×B` similarity matrix on an already-computed
   batch of pooled vectors) while directly encouraging same-class pooled
   representations to cluster — including the classes that are collapsing.

Everything else — data loading, the 70/15/15 stratified split, EDA
augmentation, speaker-role embeddings, the 3-encoder sweep, threshold
tuning, and all the reporting/plots — is unchanged from v3, so results
are directly comparable.

**Kept fast on purpose**: no hierarchical utterance encoder, no separate
contrastive-pretraining stage — those are the "go big" README options
(#2, #3-full) and would multiply training time. Everything here adds
negligible cost per batch; total runtime should stay close to v3's
~15–20 min for all three encoders.

## 0b. Install / upgrade dependencies

In [ ]:
# =============================================================
# 0b. INSTALL / UPGRADE DEPENDENCIES (online mode)
# =============================================================
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.44.0,<4.47.0", "sentencepiece", "tqdm",
    "seaborn", "scikit-learn",
], check=True)
# --no-deps: keep the GPU-matched torch already on the Kaggle image
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--no-deps", "accelerate>=0.34.0",
], check=True)
print("All required Python packages installed successfully.")

## 1. Imports & global config

In [ ]:
# =============================================================
# 1. IMPORTS & GLOBAL CONFIG
# =============================================================
import os, re, json, glob, gc, time, warnings, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                              classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"torch version: {torch.__version__} (built for CUDA {torch.version.cuda})")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---- CONFIG: the only block you should need to touch -----------------
CONFIG = {
    # Same fuzzy recursive search-root convention as v3, so this works
    # unmodified whether the labeled CSV is added as a Kaggle Dataset
    # input or dropped anywhere under these roots.
    "search_roots": ["/kaggle/input", "/kaggle/working", "/workspace", "/data", "."],

    "out_dir": "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs",
    # v4 writes its own output filenames (suffixed "_v4") and its own
    # checkpoint subdirectory, so this can run in the same out_dir as
    # capstone-v3-augmented.ipynb without clobbering v3's artifacts.
    "run_tag": "v4",

    "column_map": {
        "text": "dialogue_text",
        "binary_label": "binary_label",
        "modifier": "modifier",
    },

    "binary_classes": ["no", "yes"],
    "modifier_classes": ["none", "deferred", "conditional"],

    "encoders": [
        {"name": "roberta-base",    "hf_id": "roberta-base"},
        {"name": "deberta-v3-base", "hf_id": "microsoft/deberta-v3-base","lr": 5e-6, "batch_size": 8, "warmup_ratio": 0.10},
        {"name": "todbert",         "hf_id": "TODBERT/TOD-BERT-JNT-V1"},
    ],

    "max_length": 256,
    "train_frac": 0.70,
    "val_frac": 0.15,
    "test_frac": 0.15,

    "batch_size": 16,
    "eval_batch_size": 32,
    "num_epochs": 25,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "max_grad_norm": 1.0,

    "binary_loss_weight": 1.0,
    "modifier_loss_weight": 0.7,

    # Speaker-aware encoding (kept from v3, unchanged) -----------------
    "use_speaker_roles": True,

    # Data augmentation (EDA, kept from v3, unchanged) -----------------
    "use_augmentation": True,
    "aug_alpha": 0.15,
    "aug_num_conditional": 6,
    "aug_num_deferred": 2,
    "aug_num_no": 1,

    # ---- v4: Class-Balanced Focal Loss (replaces v3's inverse-freq
    # weighted CE + label smoothing) --------------------------------
    # "Class-Balanced Loss Based on Effective Number of Samples"
    # (Cui et al., 2019, CVPR). beta close to 1 -> weight ~ inverse
    # frequency for well-populated classes but saturates gracefully for
    # near-zero-sample classes (e.g. conditional) instead of exploding.
    "cb_beta": 0.999,
    # Focal-loss focusing parameter: down-weights already-easy/confident
    # predictions so the loss keeps pushing on hard/minority examples
    # for the whole run, not just at initialisation.
    "focal_gamma": 2.0,

    # ---- v4: Supervised Contrastive auxiliary loss --------------------
    # In-batch, single-stage (no separate pretraining pass -> cheap).
    # Pulls same-class pooled representations together / pushes apart.
    # Modifier weight is higher because that is the head that collapses
    # hardest (85.6% "none").
    "use_contrastive": True,
    "contrastive_temperature": 0.1,
    "contrastive_weight_binary": 0.10,
    "contrastive_weight_modifier": 0.30,

    "early_stopping_patience": 5,
    "checkpoint_dir_name": "checkpoints_v4",
}

os.makedirs(CONFIG["out_dir"], exist_ok=True)
os.makedirs(os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"]), exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "encoders"}, indent=2))
print("Encoders:", [m["hf_id"] for m in CONFIG["encoders"]])

## 2. Load the labeled dataset

In [ ]:
# =============================================================
# 2. DATA DISCOVERY & LOADING  (unchanged from v3)
# =============================================================

def find_files_ci(roots, must_contain_all, suffix):
    must_contain_all = [t.lower() for t in must_contain_all]
    suffix = suffix.lower()
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                low = fn.lower()
                if low.endswith(suffix) and all(tok in low for tok in must_contain_all):
                    found.append(os.path.join(dirpath, fn))
    seen, unique = set(), []
    for f in found:
        if f not in seen:
            seen.add(f)
            unique.append(f)
    return unique

csv_paths = find_files_ci(CONFIG["search_roots"], must_contain_all=["label"], suffix=".csv")
if not csv_paths:
    csv_paths = find_files_ci(CONFIG["search_roots"], must_contain_all=["manual"], suffix=".csv")

print(f"Found {len(csv_paths)} candidate CSV(s):")
for p in csv_paths:
    print(" -", p)

assert len(csv_paths) > 0, (
    "No labeled CSV found under " + str(CONFIG["search_roots"]) +
    ". Make sure the labeled dataset has been added as a Kaggle input "
    "(Notebook -> Add Input), or update CONFIG['search_roots']."
)

_dfs = [pd.read_csv(p) for p in csv_paths]
raw_df = pd.concat(_dfs, ignore_index=True) if len(_dfs) > 1 else _dfs[0]

cm = CONFIG["column_map"]
missing_cols = [c for c in cm.values() if c not in raw_df.columns]
assert not missing_cols, (
    f"Expected column(s) {missing_cols} not found in loaded CSV(s). "
    f"Columns present: {list(raw_df.columns)}. "
    "Update CONFIG['column_map'] to match the new dataset's headers."
)

df = raw_df.rename(columns={v: k for k, v in cm.items()})[list(cm.keys())].copy()

orig_id_col = None
for cand in ["conversation_id", "dialogue_id", "id"]:
    if cand in raw_df.columns:
        orig_id_col = cand
        break
if orig_id_col is not None:
    df["_orig_id"] = raw_df[orig_id_col].values
    before = len(df)
    df = df.drop_duplicates(subset="_orig_id").drop(columns="_orig_id").reset_index(drop=True)
    if len(df) != before:
        print(f"Dropped {before - len(df)} duplicate row(s) by `{orig_id_col}`.")

df["text"] = df["text"].fillna("").astype(str)
df["binary_label"] = df["binary_label"].fillna("").astype(str).str.strip().str.lower()

df["modifier"] = df["modifier"].fillna("").astype(str).str.strip().str.lower()
df.loc[df["modifier"].isin(["", "nan", "none provided", "na"]), "modifier"] = "none"

bad_binary = ~df["binary_label"].isin(CONFIG["binary_classes"])
bad_modifier = ~df["modifier"].isin(CONFIG["modifier_classes"])
if bad_binary.any() or bad_modifier.any():
    print(f"WARNING: dropping {int((bad_binary | bad_modifier).sum())} row(s) with "
          f"out-of-vocabulary binary_label/modifier values.")
    print("  bad binary_label values:", sorted(df.loc[bad_binary, "binary_label"].unique()))
    print("  bad modifier values:", sorted(df.loc[bad_modifier, "modifier"].unique()))
    df = df.loc[~(bad_binary | bad_modifier)].reset_index(drop=True)

df["binary_id"] = df["binary_label"].map({c: i for i, c in enumerate(CONFIG["binary_classes"])})
df["modifier_id"] = df["modifier"].map({c: i for i, c in enumerate(CONFIG["modifier_classes"])})

print(f"\nLoaded {len(df)} labeled rows after cleaning.")
df.head()

## 2b. Class distribution & majority-class baseline

In [ ]:
# =============================================================
# 2b. CLASS DISTRIBUTION & MAJORITY-CLASS BASELINE  (unchanged from v3)
# =============================================================
print("binary_label distribution:")
print(df["binary_label"].value_counts(), "\n")
print("modifier distribution:")
print(df["modifier"].value_counts(), "\n")
print("modifier distribution, restricted to binary_label == 'yes' rows "
      "(the population the modifier head is conditioned on):")
print(df.loc[df["binary_label"] == "yes", "modifier"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x="binary_label", order=CONFIG["binary_classes"], ax=axes[0])
axes[0].set_title("binary_label distribution")
sns.countplot(data=df, x="modifier", order=CONFIG["modifier_classes"], ax=axes[1])
axes[1].set_title("modifier distribution")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "label_distribution.png"), dpi=150)
plt.show()

def majority_baseline(y_true_ids, n_classes):
    majority_class = Counter(y_true_ids).most_common(1)[0][0]
    y_pred = [majority_class] * len(y_true_ids)
    acc = accuracy_score(y_true_ids, y_pred)
    f1_macro = f1_score(y_true_ids, y_pred, average="macro", labels=list(range(n_classes)), zero_division=0)
    return {"majority_class": majority_class, "accuracy": acc, "macro_f1": f1_macro}

baseline_binary = majority_baseline(df["binary_id"].values, len(CONFIG["binary_classes"]))
baseline_modifier = majority_baseline(df["modifier_id"].values, len(CONFIG["modifier_classes"]))

print("\nMajority-class baseline (whole dataset, for reference -- the real "
      "reported baseline further down uses the TEST split only):")
print("  binary_label:", baseline_binary)
print("  modifier    :", baseline_modifier)

## 3. Stratified 70/15/15 train/val/test split

In [ ]:
# =============================================================
# 3. STRATIFIED 70/15/15 SPLIT (by binary_label x modifier combination) -- unchanged from v3
# =============================================================

df["_stratum"] = df["binary_label"] + "_" + df["modifier"]

strat_counts = df["_stratum"].value_counts()
rare = strat_counts[strat_counts < 2].index
df.loc[df["_stratum"].isin(rare), "_stratum"] = "_singleton_bucket"

train_df, temp_df = train_test_split(
    df, test_size=(CONFIG["val_frac"] + CONFIG["test_frac"]),
    stratify=df["_stratum"], random_state=SEED,
)
temp_counts = temp_df["_stratum"].value_counts()
rare2 = temp_counts[temp_counts < 2].index
temp_df = temp_df.copy()
temp_df.loc[temp_df["_stratum"].isin(rare2), "_stratum"] = "_singleton_bucket"

rel_test_size = CONFIG["test_frac"] / (CONFIG["val_frac"] + CONFIG["test_frac"])
val_df, test_df = train_test_split(
    temp_df, test_size=rel_test_size,
    stratify=temp_df["_stratum"], random_state=SEED,
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: n={len(split)} ({len(split)/len(df):.1%})")

print("\nbinary_label composition per split:")
display(pd.concat({
    "train": train_df["binary_label"].value_counts(normalize=True),
    "val": val_df["binary_label"].value_counts(normalize=True),
    "test": test_df["binary_label"].value_counts(normalize=True),
}, axis=1))

print("\nmodifier composition per split:")
display(pd.concat({
    "train": train_df["modifier"].value_counts(normalize=True),
    "val": val_df["modifier"].value_counts(normalize=True),
    "test": test_df["modifier"].value_counts(normalize=True),
}, axis=1))

for split in (train_df, val_df, test_df):
    split.drop(columns=["_stratum"], inplace=True)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

## 3b. Data Augmentation (EDA) — TRAIN SPLIT ONLY (unchanged from v3)

In [ ]:
# =============================================================
# 3b. EDA AUGMENTATION (train split only, label-safety-guarded) -- unchanged from v3
# =============================================================
import random as _random

PROTECTED_WORDS = {
    "if", "unless", "provided", "as", "long", "condition", "conditional",
    "after", "once", "later", "when", "then", "next", "eventually",
    "promise", "will", "would", "could", "might", "may", "maybe",
    "paycheck", "payday", "month", "week", "tomorrow", "soon",
    "not", "no", "never", "don't", "dont", "won't", "wont", "can't", "cant",
    "donate", "donation", "give", "money", "charity", "pledge",
}
SPEAKER_MARKERS = {"[persuader]", "[persuadee]"}

_SYNONYMS = {
    "good": ["nice", "great", "fine", "decent"],
    "really": ["truly", "genuinely", "honestly"],
    "think": ["believe", "feel", "figure"],
    "want": ["would like", "wish", "hope"],
    "help": ["assist", "support", "aid"],
    "people": ["folks", "individuals", "persons"],
    "important": ["significant", "vital", "essential"],
    "cause": ["mission", "campaign", "effort"],
    "understand": ["see", "get", "realize"],
    "sure": ["certain", "confident", "positive"],
    "sorry": ["apologies", "regret", "my bad"],
    "okay": ["alright", "fine", "sure"],
    "thanks": ["thank you", "appreciate it", "cheers"],
    "great": ["awesome", "wonderful", "fantastic"],
    "kids": ["children", "youth", "young ones"],
    "families": ["households", "homes"],
    "problem": ["issue", "trouble", "difficulty"],
    "talk": ["chat", "speak", "discuss"],
    "guess": ["suppose", "reckon", "figure"],
    "job": ["work", "career", "position"],
    "busy": ["swamped", "occupied", "tied up"],
}

def _get_synonym(word):
    lw = word.lower()
    if lw in _SYNONYMS:
        return _random.choice(_SYNONYMS[lw])
    return None

def _tokenize_protecting_markers(text):
    return text.split(" ")

def _is_locked(tok):
    low = tok.strip(".,!?;:\"'").lower()
    return low in PROTECTED_WORDS or low in SPEAKER_MARKERS or tok.lower() in SPEAKER_MARKERS

def eda_synonym_replacement(words, n):
    new_words = words.copy()
    candidates = [i for i, w in enumerate(new_words) if not _is_locked(w) and _get_synonym(w)]
    _random.shuffle(candidates)
    replaced = 0
    for idx in candidates:
        syn = _get_synonym(new_words[idx])
        if syn:
            new_words[idx] = syn
            replaced += 1
        if replaced >= n:
            break
    return new_words

def eda_random_deletion(words, p):
    if len(words) <= 3:
        return words.copy()
    new_words = []
    for w in words:
        if _is_locked(w):
            new_words.append(w)
        elif _random.random() > p:
            new_words.append(w)
    if len(new_words) == 0:
        return [words[_random.randrange(len(words))]]
    return new_words

def eda_random_swap(words, n):
    new_words = words.copy()
    swappable = [i for i, w in enumerate(new_words) if not _is_locked(w)]
    for _ in range(n):
        if len(swappable) < 2:
            break
        i, j = _random.sample(swappable, 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words

def eda_random_insertion(words, n):
    new_words = words.copy()
    for _ in range(n):
        source_candidates = [w for w in new_words if not _is_locked(w) and _get_synonym(w)]
        if not source_candidates:
            break
        w = _random.choice(source_candidates)
        syn = _get_synonym(w)
        insert_pos = _random.randrange(len(new_words) + 1)
        new_words.insert(insert_pos, syn)
    return new_words

def eda_augment_one(text, alpha=0.15, seed=None):
    if seed is not None:
        _random.seed(seed)
    words = _tokenize_protecting_markers(text)
    n_ops = max(1, int(alpha * len(words)))

    op = _random.choice(["synonym", "swap", "delete", "insert"])
    if op == "synonym":
        words = eda_synonym_replacement(words, n_ops)
    elif op == "swap":
        words = eda_random_swap(words, n_ops)
    elif op == "delete":
        words = eda_random_deletion(words, p=alpha)
    else:
        words = eda_random_insertion(words, n_ops)

    return " ".join(words)

def augment_dataframe_for_minority_classes(df, config, seed=42):
    _random.seed(seed)
    rows_to_add = []

    yes_idx = config["binary_classes"].index("yes")
    no_idx = config["binary_classes"].index("no")
    cond_idx = config["modifier_classes"].index("conditional")
    def_idx = config["modifier_classes"].index("deferred")

    plan = [
        (df["binary_id"] == yes_idx) & (df["modifier_id"] == cond_idx), config["aug_num_conditional"],
        (df["binary_id"] == yes_idx) & (df["modifier_id"] == def_idx),  config["aug_num_deferred"],
        (df["binary_id"] == no_idx),                                   config["aug_num_no"],
    ]
    for mask, n_copies in zip(plan[0::2], plan[1::2]):
        subset = df.loc[mask]
        for _, row in subset.iterrows():
            for k in range(n_copies):
                aug_text = eda_augment_one(row["text"], alpha=config["aug_alpha"],
                                            seed=hash((row["text"], k)) % (2**31))
                new_row = row.copy()
                new_row["text"] = aug_text
                rows_to_add.append(new_row)

    if not rows_to_add:
        return df.reset_index(drop=True)

    aug_df = pd.DataFrame(rows_to_add)
    out = pd.concat([df, aug_df], ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


if CONFIG.get("use_augmentation", False):
    _before_n = len(train_df)
    _before_dist = train_df.loc[train_df["binary_label"] == "yes", "modifier"].value_counts()

    train_df = augment_dataframe_for_minority_classes(train_df, CONFIG, seed=SEED)

    _after_n = len(train_df)
    _after_dist = train_df.loc[train_df["binary_label"] == "yes", "modifier"].value_counts()

    print(f"Train split size: {_before_n} -> {_after_n} rows after EDA augmentation")
    print("\nmodifier distribution among binary=yes TRAIN rows, before -> after:")
    display(pd.concat({"before": _before_dist, "after": _after_dist}, axis=1).fillna(0).astype(int))
    print("\nbinary_label distribution after augmentation:")
    print(train_df["binary_label"].value_counts())
else:
    print("Augmentation disabled (CONFIG['use_augmentation'] = False).")

## 4. Dataset, tokenization & speaker-role IDs (unchanged from v3)

In [ ]:
# =============================================================
# 4. TORCH DATASET + SPEAKER-ROLE ID COMPUTATION -- unchanged from v3
# =============================================================
import re

_SPEAKER_ROLE_MAP = {"[Persuader]": 0, "[Persuadee]": 1}
_SPEAKER_PATTERN  = re.compile(r'\[Persuader\]|\[Persuadee\]')
ROLE_PERSUADER = 0
ROLE_PERSUADEE = 1
ROLE_SPECIAL   = 2


class DonationIntentDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.binary_ids = dataframe["binary_id"].tolist()
        self.modifier_ids = dataframe["modifier_id"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "binary_id": self.binary_ids[idx],
            "modifier_id": self.modifier_ids[idx],
        }


def compute_batch_role_ids(texts, offset_mapping):
    batch_role_ids = []

    for text, token_offsets in zip(texts, offset_mapping.tolist()):
        boundaries = sorted(
            [(m.start(), _SPEAKER_ROLE_MAP[m.group()])
             for m in _SPEAKER_PATTERN.finditer(text)],
            key=lambda x: x[0],
        )

        role_ids = []
        for char_start, char_end in token_offsets:
            if char_start == 0 and char_end == 0:
                role_ids.append(ROLE_SPECIAL)
                continue

            current_role = ROLE_SPECIAL
            for bpos, brole in boundaries:
                if bpos <= char_start:
                    current_role = brole
                else:
                    break
            role_ids.append(current_role)

        batch_role_ids.append(role_ids)

    return torch.tensor(batch_role_ids, dtype=torch.long)


def make_collate_fn(tokenizer, max_length, use_speaker_roles=False):
    def collate_fn(batch):
        texts = [b["text"] for b in batch]

        if use_speaker_roles:
            try:
                enc = tokenizer(
                    texts, padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt",
                    return_offsets_mapping=True,
                )
                offset_mapping = enc.pop("offset_mapping")
                enc["role_ids"] = compute_batch_role_ids(texts, offset_mapping)
            except Exception as exc:
                print(f"  [WARNING] offset_mapping not available, "
                      f"speaker roles disabled for this batch: {exc}")
                enc = tokenizer(
                    texts, padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt",
                )
        else:
            enc = tokenizer(
                texts, padding=True, truncation=True,
                max_length=max_length, return_tensors="pt",
            )

        enc["binary_labels"]   = torch.tensor([b["binary_id"]   for b in batch], dtype=torch.long)
        enc["modifier_labels"] = torch.tensor([b["modifier_id"] for b in batch], dtype=torch.long)
        return enc
    return collate_fn

## 5. Model architecture — attention pooling + cascaded heads (v4)

```
Dialogue text
   |
   Tokenizer (AutoTokenizer, max_len=256, return_offsets_mapping=True)
   |
Shared Transformer Encoder  (RoBERTa-base | DeBERTa-v3-base | TOD-BERT)
   |
last_hidden_state  [B, L, H]
   |
   +--(+)-- Speaker-Role Embedding  nn.Embedding(3, H)      [unchanged from v3]
   |
   +----------------------------+----------------------------+
   |                            |
[CLS]/<s> vector           Attention pooling                  <- NEW (replaces mean-pool)
   [B, H]                  score_t = v^T tanh(W h_t)
   |                       weights = softmax_t(score_t | non-pad)
   |                       pooled  = sum_t weights_t * h_t   [B, H]
   +----------------------------+----------------------------+
                       concat -> Linear(2H,H) -> GELU -> Dropout
                                       |
                                 shared_repr [B, H]
                                       |
                          +------------+------------+
                          |                          |
                     binary_head               modifier_head          <- NEW: cascaded
                (Linear, 2-cls)     input = [shared_repr ; softmax(binary_logits).detach()]
                          |          Linear(H+2, H/2) -> GELU -> Dropout -> Linear(H/2, 3)
                    binary_logits                  modifier_logits
```

Why each change:
- **Attention pooling instead of mean-pool.** Mean-pooling weights every
  token equally, so on a long dialogue the handful of tokens that actually
  signal `deferred`/`conditional` (`"if"`, `"next month"`, hedging
  language) get averaged down to near-nothing. A learned per-token score
  lets the model concentrate the pooled vector on the tokens that matter.
- **Fused with the `[CLS]` vector**, not a replacement for it — keeps the
  encoder's own sequence-summary signal alongside the token-weighted one.
- **Cascaded modifier head.** The task is structurally two-stage: modifier
  only means something when `binary_label == "yes"`. v3's modifier head
  only ever saw this indirectly, through the *masked loss* (computed only
  on yes-rows) — the head itself had no explicit signal about the model's
  own binary decision. Feeding `softmax(binary_logits).detach()` in gives
  it that signal directly, without letting modifier-loss gradients corrupt
  the binary head (hence `.detach()`).

In [ ]:
# =============================================================
# 5. ATTENTION-POOLING + CASCADED TWO-HEAD MODEL  (v4)
# =============================================================

class AttentionPoolingCascadedClassifier(nn.Module):
    """
    Multi-task classifier with learned attention pooling and a cascaded
    modifier head, replacing v3's masked-mean-pool + independent-heads design.

    Architecture
    ------------
    1. Shared transformer encoder (RoBERTa / DeBERTa-v3 / TOD-BERT via AutoModel).
    2. Speaker-role embedding [optional, unchanged from v3]: nn.Embedding(3, H)
       added element-wise to last_hidden_state before pooling.
    3. Two pooled views, fused:
       - the `[CLS]`/`<s>` vector (last_hidden_state[:, 0, :])
       - a learned additive-attention pooled vector over all non-pad tokens
       concatenated -> Linear(2H, H) -> GELU -> Dropout -> shared_repr.
    4. binary_head: Linear(H, n_binary) on shared_repr.
    5. modifier_head: takes [shared_repr ; softmax(binary_logits).detach()]
       through a small 2-layer MLP -- explicit access to the model's own
       binary decision, matching the task's conditional structure.

    Returns (binary_logits, modifier_logits, shared_repr). shared_repr is
    exposed so the training loop can apply a supervised-contrastive
    auxiliary loss directly on the pooled representation.
    """

    NUM_ROLES = 3  # Persuader / Persuadee / special-padding-unknown

    def __init__(self, hf_id, n_binary, n_modifier, dropout=0.1,
                 use_speaker_roles=False):
        super().__init__()
        self.use_speaker_roles = use_speaker_roles
        self.encoder = AutoModel.from_pretrained(hf_id)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)

        if use_speaker_roles:
            self.role_embedding = nn.Embedding(self.NUM_ROLES, hidden)
            nn.init.normal_(self.role_embedding.weight, std=0.02)
        else:
            self.role_embedding = None

        # Additive ("Bahdanau-style") attention pooling.
        self.attn_proj  = nn.Linear(hidden, hidden)
        self.attn_score = nn.Linear(hidden, 1, bias=False)

        self.fuse = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.binary_head = nn.Linear(hidden, n_binary)
        self.modifier_head = nn.Sequential(
            nn.Linear(hidden + n_binary, max(hidden // 2, n_modifier)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(max(hidden // 2, n_modifier), n_modifier),
        )

    def _attention_pool(self, last_hidden, attention_mask):
        """Learned per-token attention pooling over non-pad tokens."""
        scores = self.attn_score(torch.tanh(self.attn_proj(last_hidden))).squeeze(-1)  # [B, L]
        scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)  # [B, L, 1]
        return (last_hidden * weights).sum(dim=1)

    def forward(self, input_ids, attention_mask, token_type_ids=None,
                role_ids=None, **_):
        import inspect
        enc_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        fwd_params = inspect.signature(self.encoder.forward).parameters
        if token_type_ids is not None and "token_type_ids" in fwd_params:
            enc_kwargs["token_type_ids"] = token_type_ids

        outputs     = self.encoder(**enc_kwargs)
        last_hidden = outputs.last_hidden_state          # [B, L, H]

        if self.use_speaker_roles and role_ids is not None:
            role_embeds = self.role_embedding(role_ids)  # [B, L, H]
            last_hidden = last_hidden + role_embeds

        cls_vec  = last_hidden[:, 0, :]
        attn_vec = self._attention_pool(last_hidden, attention_mask)
        shared   = self.dropout(self.fuse(torch.cat([cls_vec, attn_vec], dim=-1)))

        binary_logits = self.binary_head(shared)
        modifier_input = torch.cat(
            [shared, torch.softmax(binary_logits, dim=-1).detach()], dim=-1)
        modifier_logits = self.modifier_head(modifier_input)

        return binary_logits, modifier_logits, shared

## 6. Loss — Class-Balanced Focal Loss + Supervised Contrastive (v4)

Replaces v3's `CrossEntropyLoss(weight=inverse_freq, label_smoothing=0.1)`
on both heads.

**Class-Balanced Focal Loss** (Cui et al., 2019, CVPR): class weights come
from the *effective number of samples* `(1 - beta^n) / (1 - beta)` rather
than raw inverse frequency — this saturates gracefully for near-zero-sample
classes like `conditional` (~9-13 real train rows) instead of assigning an
enormous, unstable weight. On top of that, the focal term
`(1-p_t)^gamma` keeps discounting *already-easy* predictions for the whole
run, so a model that starts predicting the majority class confidently
doesn't get to coast on a fixed loss scale the way plain weighted CE
allows.

**Supervised Contrastive loss** (Khosla et al., 2020), computed in-batch
on `shared_repr`: for each anchor with at least one same-class partner in
the batch, pulls same-class pooled vectors together and pushes different-
class ones apart via a batch similarity matrix. This is a single-stage
stand-in for README idea #3 (full SimCSE needs an unsupervised
pretraining stage) — it adds one `B x B` matmul on vectors already
computed for the forward pass, so it is effectively free compute-wise.
Applied to binary labels (small weight) and, more heavily, to modifier
labels restricted to `binary_label == "yes"` rows (the class that
collapses hardest).

In [ ]:
# =============================================================
# 6. CLASS-BALANCED FOCAL LOSS + SUPERVISED CONTRASTIVE LOSS  (v4)
# =============================================================

class ClassBalancedFocalLoss(nn.Module):
    """Cui et al. (2019) class-balanced re-weighting (effective number of
    samples) combined with the focal-loss focusing term (Lin et al., 2017).
    """
    def __init__(self, samples_per_class, beta=0.999, gamma=2.0):
        super().__init__()
        samples_per_class = np.asarray(samples_per_class, dtype=np.float64)
        samples_per_class = np.clip(samples_per_class, 1.0, None)  # avoid beta**0 edge case
        effective_num = 1.0 - np.power(beta, samples_per_class)
        effective_num = np.where(effective_num <= 0, 1e-8, effective_num)
        weights = (1.0 - beta) / effective_num
        weights = weights / weights.sum() * len(samples_per_class)
        self.register_buffer("class_weights", torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, logits, targets):
        if logits.size(0) == 0:
            return torch.zeros((), device=logits.device)
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        pt     = probs.gather(1, targets.unsqueeze(1)).squeeze(1).clamp(min=1e-8)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        alpha_t = self.class_weights.to(logits.device)[targets]
        loss = -alpha_t * (1 - pt).pow(self.gamma) * log_pt
        return loss.mean()


def supervised_contrastive_loss(pooled, labels, temperature=0.1):
    """
    In-batch supervised contrastive loss (Khosla et al., 2020), single-view.
    For each anchor with >=1 same-class partner in the batch, pulls its
    representation toward same-class partners and away from the rest.
    Anchors with zero same-class partners in the batch contribute nothing
    (common in small batches with rare classes -- this is expected).
    """
    n = pooled.size(0)
    if n < 2:
        return torch.zeros((), device=pooled.device)

    z = F.normalize(pooled, dim=-1)
    sim = torch.matmul(z, z.T) / temperature                    # [B, B]

    labels = labels.view(-1, 1)
    same_class  = (labels == labels.T).float()
    self_mask   = torch.eye(n, device=z.device)
    positive_mask = same_class - self_mask

    logits_max, _ = sim.max(dim=1, keepdim=True)
    logits = sim - logits_max.detach()
    exp_logits = torch.exp(logits) * (1 - self_mask)
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

    pos_counts = positive_mask.sum(dim=1)
    valid = pos_counts > 0
    if valid.sum() == 0:
        return torch.zeros((), device=pooled.device)

    mean_log_prob_pos = (positive_mask * log_prob).sum(dim=1)[valid] / pos_counts[valid]
    return -mean_log_prob_pos.mean()


def samples_per_class(ids, n_classes):
    counts = np.bincount(ids, minlength=n_classes).astype(float)
    counts[counts == 0] = 1.0  # absent-in-train class: treat as 1 to avoid div issues
    return counts

binary_samples_per_class = samples_per_class(train_df["binary_id"].values, len(CONFIG["binary_classes"]))

# Modifier counts from yes-only training rows -- the modifier task is
# conditioned on binary_label="yes" (unchanged reasoning from v3).
train_yes = train_df[train_df["binary_label"] == "yes"].reset_index(drop=True)
modifier_samples_per_class = samples_per_class(train_yes["modifier_id"].values, len(CONFIG["modifier_classes"]))

print("binary_label samples/class (train):", dict(zip(CONFIG["binary_classes"], binary_samples_per_class.tolist())))
print("modifier samples/class (train, yes-only):", dict(zip(CONFIG["modifier_classes"], modifier_samples_per_class.tolist())))
print(f"  (computed from {len(train_yes)} yes-only / {len(train_df)} total training rows)")

binary_focal_loss_fn = ClassBalancedFocalLoss(
    binary_samples_per_class, beta=CONFIG["cb_beta"], gamma=CONFIG["focal_gamma"])
modifier_focal_loss_fn = ClassBalancedFocalLoss(
    modifier_samples_per_class, beta=CONFIG["cb_beta"], gamma=CONFIG["focal_gamma"])

print("\nClass-balanced weights (effective-number-of-samples, beta="
      f"{CONFIG['cb_beta']}):")
print("  binary:  ", dict(zip(CONFIG["binary_classes"], binary_focal_loss_fn.class_weights.tolist())))
print("  modifier:", dict(zip(CONFIG["modifier_classes"], modifier_focal_loss_fn.class_weights.tolist())))

## 7. Training / eval functions (v4)

Same overall shape as v3 (threshold tuning, early stopping, best-checkpoint
reload) with three changes:

1. `evaluate()` now also reports **binary macro-F1** (average of `no`-F1
   and `yes`-F1), alongside the yes-only F1 kept for continuity with v3's
   reporting.
2. **Checkpoint selection / early stopping uses binary macro-F1**, not
   yes-only F1 — this is the bug fix described in the intro cell.
3. The training step now computes class-balanced focal losses plus the
   in-batch supervised-contrastive terms (binary + yes-conditioned
   modifier), gated by `CONFIG["use_contrastive"]`.

In [ ]:
# =============================================================
# 7. TRAIN / EVAL FUNCTIONS  (v4: focal + contrastive loss, macro-F1 selection)
# =============================================================

def compute_losses(binary_logits, modifier_logits, shared, binary_labels, modifier_labels,
                    yes_idx, include_contrastive):
    """Shared loss computation used by both the training step and evaluate()."""
    binary_loss = binary_focal_loss_fn(binary_logits, binary_labels)

    yes_mask = binary_labels == yes_idx
    if yes_mask.any():
        mod_loss = modifier_focal_loss_fn(modifier_logits[yes_mask], modifier_labels[yes_mask])
    else:
        mod_loss = torch.zeros((), device=binary_logits.device)

    total = CONFIG["binary_loss_weight"] * binary_loss + CONFIG["modifier_loss_weight"] * mod_loss

    if include_contrastive and CONFIG.get("use_contrastive", False):
        con_binary = supervised_contrastive_loss(
            shared, binary_labels, temperature=CONFIG["contrastive_temperature"])
        if yes_mask.sum() >= 2:
            con_modifier = supervised_contrastive_loss(
                shared[yes_mask], modifier_labels[yes_mask],
                temperature=CONFIG["contrastive_temperature"])
        else:
            con_modifier = torch.zeros((), device=binary_logits.device)
        total = (total
                 + CONFIG["contrastive_weight_binary"] * con_binary
                 + CONFIG["contrastive_weight_modifier"] * con_modifier)

    return total


def evaluate(model, loader, device, binary_threshold=0.5):
    """
    Evaluate model on a DataLoader.

    binary_threshold : float
        P(yes) >= binary_threshold -> predict 'yes'.
    """
    model.eval()
    all_binary_true, all_binary_pred = [], []
    all_binary_probs = []
    all_modifier_true, all_modifier_pred = [], []
    total_loss = 0.0
    n_batches  = 0
    yes_idx    = CONFIG["binary_classes"].index("yes")

    with torch.no_grad():
        for batch in loader:
            binary_labels   = batch.pop("binary_labels").to(device)
            modifier_labels = batch.pop("modifier_labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}

            binary_logits, modifier_logits, shared = model(**batch)

            # Eval loss reported WITHOUT the contrastive term (it's a
            # representation-shaping regulariser for training, not a
            # meaningful "how good are these predictions" number).
            loss = compute_losses(binary_logits, modifier_logits, shared,
                                   binary_labels, modifier_labels, yes_idx,
                                   include_contrastive=False)
            total_loss += loss.item()
            n_batches  += 1

            binary_probs_yes = torch.softmax(binary_logits, dim=-1)[:, yes_idx]
            binary_preds     = (binary_probs_yes >= binary_threshold).long()

            all_binary_true.extend(binary_labels.cpu().tolist())
            all_binary_pred.extend(binary_preds.cpu().tolist())
            all_binary_probs.extend(binary_probs_yes.cpu().tolist())
            all_modifier_true.extend(modifier_labels.cpu().tolist())
            all_modifier_pred.extend(modifier_logits.argmax(dim=-1).cpu().tolist())

    binary_acc       = accuracy_score(all_binary_true, all_binary_pred)
    binary_f1        = f1_score(all_binary_true, all_binary_pred, average="binary",
                                pos_label=yes_idx, zero_division=0)
    # v4: macro-F1 across BOTH binary classes -- this is what checkpoint
    # selection uses (see train_one_encoder below). A model that always
    # predicts "yes" scores well on `binary_f1` above but poorly here,
    # because "no" recall collapses to 0.
    binary_macro_f1  = f1_score(all_binary_true, all_binary_pred, average="macro",
                                 labels=[0, 1], zero_division=0)
    modifier_macro_f1 = f1_score(all_modifier_true, all_modifier_pred, average="macro",
                                  labels=list(range(len(CONFIG["modifier_classes"]))),
                                  zero_division=0)

    yes_idx_eval = [i for i, b in enumerate(all_binary_true) if b == yes_idx]
    if yes_idx_eval:
        mod_true_yes = [all_modifier_true[i] for i in yes_idx_eval]
        mod_pred_yes = [all_modifier_pred[i] for i in yes_idx_eval]
        modifier_macro_f1_yes = f1_score(
            mod_true_yes, mod_pred_yes, average="macro",
            labels=list(range(len(CONFIG["modifier_classes"]))), zero_division=0)
    else:
        modifier_macro_f1_yes = 0.0

    return {
        "loss":                  total_loss / max(n_batches, 1),
        "binary_accuracy":       binary_acc,
        "binary_f1":             binary_f1,
        "binary_macro_f1":       binary_macro_f1,
        "modifier_macro_f1":     modifier_macro_f1,
        "modifier_macro_f1_yes": modifier_macro_f1_yes,
        "binary_true":    all_binary_true,  "binary_pred":    all_binary_pred,
        "binary_probs":   all_binary_probs,
        "modifier_true":  all_modifier_true, "modifier_pred":  all_modifier_pred,
    }


def find_optimal_binary_threshold(model, val_loader, device):
    """Grid-search for the binary threshold that maximises macro-F1 on val.
    Unchanged from v3 -- this part already used macro-F1."""
    model.eval()
    all_true, all_probs = [], []
    yes_idx = CONFIG["binary_classes"].index("yes")

    with torch.no_grad():
        for batch in val_loader:
            binary_labels = batch.pop("binary_labels").to(device)
            batch.pop("modifier_labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            binary_logits, _, _ = model(**batch)
            probs = torch.softmax(binary_logits, dim=-1)[:, yes_idx]
            all_true.extend(binary_labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    best_threshold, best_macro_f1 = 0.5, 0.0
    for t in [v / 100 for v in range(10, 91, 5)]:
        preds   = [1 if p >= t else 0 for p in all_probs]
        macro_f1 = f1_score(all_true, preds, average="macro",
                             labels=[0, 1], zero_division=0)
        if macro_f1 > best_macro_f1:
            best_macro_f1, best_threshold = macro_f1, t

    return best_threshold, best_macro_f1


def train_one_encoder(encoder_cfg, train_df, val_df, test_df):
    name, hf_id    = encoder_cfg["name"], encoder_cfg["hf_id"]
    use_roles      = CONFIG.get("use_speaker_roles", False)
    sep_line       = "=" * 70
    print(f"\n{sep_line}\nTraining {name} ({hf_id})"
          f"  [speaker_roles={'ON' if use_roles else 'OFF'}]"
          f"  [contrastive={'ON' if CONFIG.get('use_contrastive') else 'OFF'}]\n{sep_line}")

    tokenizer  = AutoTokenizer.from_pretrained(hf_id)
    collate_fn = make_collate_fn(tokenizer, CONFIG["max_length"],
                                  use_speaker_roles=use_roles)

    train_ds = DonationIntentDataset(train_df)
    val_ds   = DonationIntentDataset(val_df)
    test_ds  = DonationIntentDataset(test_df)

    enc_batch_size = encoder_cfg.get("batch_size", CONFIG["batch_size"])
    train_loader = DataLoader(train_ds, batch_size=enc_batch_size,
                               shuffle=True, collate_fn=collate_fn)
    val_loader  = DataLoader(val_ds,  batch_size=CONFIG["eval_batch_size"],
                              shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["eval_batch_size"],
                              shuffle=False, collate_fn=collate_fn)

    model = AttentionPoolingCascadedClassifier(
        hf_id,
        n_binary=len(CONFIG["binary_classes"]),
        n_modifier=len(CONFIG["modifier_classes"]),
        use_speaker_roles=use_roles,
    ).to(DEVICE)

    yes_idx = CONFIG["binary_classes"].index("yes")

    encoder_lr = encoder_cfg.get("lr", CONFIG["lr"])
    optimizer  = torch.optim.AdamW(model.parameters(), lr=encoder_lr,
                                    weight_decay=CONFIG["weight_decay"], eps=1e-6)
    total_steps = len(train_loader) * CONFIG["num_epochs"]
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * encoder_cfg.get("warmup_ratio", CONFIG["warmup_ratio"])),
        num_training_steps=total_steps,
    )

    ckpt_path            = os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"],
                                        f"{name}_best.pt")
    best_val_score       = -1.0
    epochs_without_improve = 0
    history              = []

    for epoch in range(1, CONFIG["num_epochs"] + 1):
        model.train()
        running_loss = 0.0
        _n = CONFIG["num_epochs"]
        pbar = tqdm(train_loader, desc=f"[{name}] epoch {epoch}/{_n}")

        for batch in pbar:
            binary_labels   = batch.pop("binary_labels").to(DEVICE)
            modifier_labels = batch.pop("modifier_labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            optimizer.zero_grad()
            binary_logits, modifier_logits, shared = model(**batch)

            loss = compute_losses(binary_logits, modifier_logits, shared,
                                   binary_labels, modifier_labels, yes_idx,
                                   include_contrastive=True)

            if torch.isnan(loss):
                print("  [WARNING] NaN loss detected -- skipping batch.")
                optimizer.zero_grad()
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        val_metrics = evaluate(model, val_loader, DEVICE, binary_threshold=0.5)
        # v4 FIX: select on binary MACRO-F1 (both classes), not yes-only F1
        # (v3's selection metric rewarded always-predict-yes collapse).
        val_score   = (val_metrics["binary_macro_f1"] + val_metrics["modifier_macro_f1_yes"]) / 2
        history.append({
            "epoch":                     epoch,
            "train_loss":                running_loss / len(train_loader),
            "val_loss":                  val_metrics["loss"],
            "val_binary_f1":             val_metrics["binary_f1"],
            "val_binary_macro_f1":       val_metrics["binary_macro_f1"],
            "val_binary_accuracy":       val_metrics["binary_accuracy"],
            "val_modifier_macro_f1":     val_metrics["modifier_macro_f1"],
            "val_modifier_macro_f1_yes": val_metrics["modifier_macro_f1_yes"],
        })
        _tl = history[-1]["train_loss"]
        _bmf = val_metrics["binary_macro_f1"]
        _mf = val_metrics["modifier_macro_f1_yes"]
        print(f"  epoch {epoch}: train_loss={_tl:.4f} "
              f"val_binary_macroF1={_bmf:.4f} val_modifier_macroF1|yes={_mf:.4f}")

        if val_score > best_val_score:
            best_val_score       = val_score
            epochs_without_improve = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> new best (avg macro-F1={val_score:.4f}), checkpoint saved.")
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= CONFIG["early_stopping_patience"]:
                print(f"  -> no improvement for {epochs_without_improve} epochs, stopping early.")
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    best_thresh, thresh_val_macro_f1 = find_optimal_binary_threshold(
        model, val_loader, DEVICE)
    print(f"\n  Threshold search -> best thresh={best_thresh:.2f}"
          f"  (val macro-F1={thresh_val_macro_f1:.4f})")

    test_m_default = evaluate(model, test_loader, DEVICE, binary_threshold=0.5)
    test_m_tuned   = evaluate(model, test_loader, DEVICE, binary_threshold=best_thresh)

    for tm in (test_m_default, test_m_tuned):
        yes_list = [i for i, t in enumerate(tm["binary_true"]) if t == yes_idx]
        if yes_list:
            mt = [tm["modifier_true"][i] for i in yes_list]
            mp = [tm["modifier_pred"][i] for i in yes_list]
            tm["modifier_macro_f1_given_binary_yes"] = f1_score(
                mt, mp, average="macro",
                labels=list(range(len(CONFIG["modifier_classes"]))), zero_division=0)
        else:
            tm["modifier_macro_f1_given_binary_yes"] = 0.0

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "name":                              name,
        "hf_id":                             hf_id,
        "history":                           history,
        "test_metrics":                      test_m_tuned,
        "test_metrics_default":              test_m_default,
        "test_binary_accuracy":              test_m_tuned["binary_accuracy"],
        "test_binary_f1":                    test_m_tuned["binary_f1"],
        "test_binary_macro_f1":              test_m_tuned["binary_macro_f1"],
        "test_modifier_macro_f1":            test_m_tuned["modifier_macro_f1"],
        "test_modifier_macro_f1_given_yes":  test_m_tuned["modifier_macro_f1_given_binary_yes"],
        "best_threshold":                    best_thresh,
        "ckpt_path":                         ckpt_path,
    }

## 8. Run training for all three encoders

In [ ]:
# =============================================================
# 8. RUN ALL THREE ENCODERS
# =============================================================
results = {}
for encoder_cfg in CONFIG["encoders"]:
    results[encoder_cfg["name"]] = train_one_encoder(encoder_cfg, train_df, val_df, test_df)

print("\nDone training all encoders:", list(results.keys()))

In [ ]:
# =============================================================
# 8b. THRESHOLD COMPARISON TABLE
# =============================================================
print(f"\n{'='*70}")
print("Binary head: default threshold (0.5) vs val-tuned threshold")
print(f"{'='*70}")
header = f"{'Model':<20} {'Thresh':>6}  {'Binary Acc':>10}  {'Bin MacroF1':>11}  "
header += f"{'no F1':>8}  {'yes F1':>8}  {'Mod F1|yes':>11}"
print(header)
print("-" * len(header))

for name, r in results.items():
    for label, tm, thresh in [
        ("(default)", r["test_metrics_default"], 0.5),
        ("(tuned)",   r["test_metrics"],         r["best_threshold"]),
    ]:
        from sklearn.metrics import f1_score as _f1
        no_f1  = _f1(tm["binary_true"], tm["binary_pred"], pos_label=0,
                      average="binary", zero_division=0)
        yes_f1 = _f1(tm["binary_true"], tm["binary_pred"], pos_label=1,
                      average="binary", zero_division=0)
        row = (f"{name+' '+label:<20} {thresh:>6.2f}  "
               f"{tm['binary_accuracy']:>10.4f}  {tm['binary_macro_f1']:>11.4f}  "
               f"{no_f1:>8.4f}  {yes_f1:>8.4f}  "
               f"{tm['modifier_macro_f1_given_binary_yes']:>11.4f}")
        print(row)
    print()

## 9. Test-set majority-class baseline (matched to the actual test split)

In [ ]:
# =============================================================
# 9. TEST-SET MAJORITY-CLASS BASELINE
# =============================================================
train_majority_binary = Counter(train_df["binary_id"]).most_common(1)[0][0]
train_majority_modifier = Counter(train_df["modifier_id"]).most_common(1)[0][0]

test_binary_true = test_df["binary_id"].values
test_modifier_true = test_df["modifier_id"].values

baseline_test_binary_pred = [train_majority_binary] * len(test_df)
baseline_test_modifier_pred = [train_majority_modifier] * len(test_df)

test_yes_mask = [b == CONFIG["binary_classes"].index("yes") for b in test_binary_true]
test_modifier_true_yes   = [test_modifier_true[i]          for i, m in enumerate(test_yes_mask) if m]
baseline_modifier_pred_yes = [baseline_test_modifier_pred[i] for i, m in enumerate(test_yes_mask) if m]

baseline_row = {
    "model": "majority_baseline",
    "binary_accuracy": accuracy_score(test_binary_true, baseline_test_binary_pred),
    "binary_f1": f1_score(test_binary_true, baseline_test_binary_pred, average="binary",
                           pos_label=CONFIG["binary_classes"].index("yes"), zero_division=0),
    "binary_macro_f1": f1_score(test_binary_true, baseline_test_binary_pred, average="macro",
                                 labels=[0, 1], zero_division=0),
    "modifier_macro_f1": f1_score(test_modifier_true, baseline_test_modifier_pred, average="macro",
                                   labels=list(range(len(CONFIG["modifier_classes"]))), zero_division=0),
    "modifier_macro_f1_given_binary_yes": f1_score(
        test_modifier_true_yes, baseline_modifier_pred_yes,
        average="macro",
        labels=list(range(len(CONFIG["modifier_classes"]))), zero_division=0),
}
print("Majority-class baseline on TEST split:")
print(json.dumps(baseline_row, indent=2))

## 10. Results summary table — v4 vs. majority baseline

In [ ]:
# =============================================================
# 10. RESULTS SUMMARY
# =============================================================
summary_rows = [baseline_row]
for name, r in results.items():
    summary_rows.append({
        "model": name,
        "binary_accuracy": r["test_binary_accuracy"],
        "binary_f1": r["test_binary_f1"],
        "binary_macro_f1": r["test_binary_macro_f1"],
        "modifier_macro_f1": r["test_modifier_macro_f1"],
        "modifier_macro_f1_given_binary_yes": r["test_modifier_macro_f1_given_yes"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(CONFIG["out_dir"], f"classifier_test_results_{CONFIG['run_tag']}.csv"), index=False)
display(summary_df)

### 10a. Bar chart — accuracy/F1 comparison across encoders + baseline

In [ ]:
plot_df = summary_df.melt(
    id_vars="model",
    value_vars=["binary_accuracy", "binary_macro_f1", "modifier_macro_f1"],
    var_name="metric", value_name="score",
)
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, x="metric", y="score", hue="model")
plt.ylim(0, 1)
plt.title("v4 classifier performance vs. majority-class baseline (test split)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], f"classifier_comparison_bar_{CONFIG['run_tag']}.png"), dpi=150)
plt.show()

### 10b. Confusion matrices — binary_label, per encoder

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5.5 * len(results), 4.5))
if len(results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(r["test_metrics"]["binary_true"], r["test_metrics"]["binary_pred"],
                           labels=list(range(len(CONFIG["binary_classes"]))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CONFIG["binary_classes"], yticklabels=CONFIG["binary_classes"], ax=ax)
    ax.set_title(f"{name} -- binary_label")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], f"confusion_matrix_binary_{CONFIG['run_tag']}.png"), dpi=150)
plt.show()

### 10c. Confusion matrices — modifier, per encoder

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5.5 * len(results), 4.5))
if len(results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(r["test_metrics"]["modifier_true"], r["test_metrics"]["modifier_pred"],
                           labels=list(range(len(CONFIG["modifier_classes"]))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
                xticklabels=CONFIG["modifier_classes"], yticklabels=CONFIG["modifier_classes"], ax=ax)
    ax.set_title(f"{name} -- modifier")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], f"confusion_matrix_modifier_{CONFIG['run_tag']}.png"), dpi=150)
plt.show()

### 10d. Training curves — val F1 per epoch, per encoder

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, r in results.items():
    hist = pd.DataFrame(r["history"])
    axes[0].plot(hist["epoch"], hist["val_binary_macro_f1"], marker="o", label=name)
    axes[1].plot(hist["epoch"], hist["val_modifier_macro_f1"], marker="o", label=name)
axes[0].set_title("Validation binary_label macro-F1 (selection metric)"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation modifier macro-F1"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], f"training_curves_{CONFIG['run_tag']}.png"), dpi=150)
plt.show()

## 10e. Classification reports (full precision/recall/F1 per class)

In [ ]:
for name, r in results.items():
    _sep = "=" * 70
    print(f"\n{_sep}\n{name} -- binary_label classification report\n{_sep}")
    print(classification_report(r["test_metrics"]["binary_true"], r["test_metrics"]["binary_pred"],
                                 target_names=CONFIG["binary_classes"], zero_division=0))
    print(f"{name} -- modifier classification report")
    print(classification_report(r["test_metrics"]["modifier_true"], r["test_metrics"]["modifier_pred"],
                                 target_names=CONFIG["modifier_classes"], zero_division=0))

## 11. Summary

v4 changes vs. `capstone-v3-augmented.ipynb` (kept as-is, this is a
separate file so both can be run/compared side by side):

| Change | Where | Why |
|---|---|---|
| Checkpoint selection uses binary **macro**-F1, not yes-only F1 | training loop | v3's selection metric rewarded always-predicting-yes; this is the most likely direct cause of the reported "labels everything yes/none" behaviour |
| Class-Balanced Focal Loss (effective # of samples + focal term) | loss | Static inverse-freq weights, even combined with label smoothing, weren't enough to stop majority-collapse in v3; focal's per-example modulation keeps pressure on hard/minority examples throughout training |
| Attention pooling (fused with CLS) replaces mean-pool | model | Mean-pooling dilutes the few tokens that carry modifier signal across a whole dialogue; learned attention lets the model find them |
| Cascaded modifier head (conditioned on binary_logits) | model | Makes the task's actual binary->modifier dependency explicit to the modifier head, not just implicit via masked loss |
| Supervised Contrastive auxiliary loss (in-batch, single-stage) | loss | Lightweight stand-in for README idea #3 (full SimCSE needs a separate pretraining stage); directly shapes the representation space to separate classes, including collapsing ones |

Unchanged from v3 (kept because they weren't the problem): speaker-role
encoding, EDA augmentation, the 70/15/15 stratified split, val-tuned
binary threshold search, the 3-encoder benchmark, and all reporting/plots.

**Still true from v3's known limitation**: `conditional` has only
~9-13 real training examples. Augmentation and the changes above give the
model more/better gradient signal per example, not new information — a
`conditional` F1 improvement here is a mitigation, not a solved problem.
The durable fix is still more labeled `conditional` examples.

**If v4 still under-performs on `conditional` specifically**, the next
things to try from the README, in order of effort: ensembling the 3
already-trained encoders' probabilities (idea #6 — cheap, reuses these
checkpoints, no retraining), then the hierarchical utterance encoder
(idea #2 — bigger structural change, addresses 256-token truncation
losing later dialogue turns, needs more of a compute budget than the
"~20-30 min" used here).